In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
import my_data_manager as mdm

cat = mdm.load_cat("galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)


/mnt/c/Users/gasep/OneDrive/Documentos/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/mnt/c/Users/gasep/OneDrive/Documentos/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [3]:
print(clusters[0].columns)
print(f"Numebr of Clusters found: {len(clusters)}")

Index(['galaxyId', 'haloId', 'pos_x', 'pos_y', 'pos_z', 'vel_x', 'vel_y',
       'vel_z', 'redshift', 'snapshot', 'sfr', 'm_star', 'm_coldgas',
       'Z_coldgas', 'R_coldgas', 'sfh_bin', 'sfh_nbin', 'RA', 'DEC', 'z_geo',
       'd_comoving', 'z_app', 'mag_u', 'mag_g', 'mag_r', 'mag_i', 'mag_z',
       'z_phot', 'firstHaloInFOFGroupId', 'log(m_200)'],
      dtype='object', name=0)
Numebr of Clusters found: 731


In [4]:
clean_clusters = [df[df.groupby("haloId")["haloId"].transform("count") > 3].reset_index(drop=True) for df in clusters]

In [5]:
import pandas as pd

N = 21

splus_clusters = [df[pd.to_numeric(df['mag_r'], errors = 'coerce') >= N].copy() for df in clusters]

In [6]:
cluster_samples = {
    "raw": clusters,
    "clean": clean_clusters,
    "splus": splus_clusters
}

## Clustering

In [7]:
import clustering_methods as clustering
import numpy as np

algorithms = {}


#algorithms['GMM'] = clustering.run_GMM
#algorithms['DBSCAN'] = clustering.run_DBSCAN
#algorithms['HDBSCAN'] = clustering.run_HDBSCAN
#algorithms['Optics'] = clustering.run_OPTICS
algorithms['Kmeans'] = clustering.run_Kmeans
#algorithms['Aglomerative'] = clustering.run_Aglomerative_Clustering
#algorithms['Affinity'] = clustering.run_Affinity_Propagation

params = {
    "max_clusters" : 0.2,
    "covariance_type" : 'full',
    "min_cluster_size" : 4,
    "min_eps" : 0.5,
    "max_eps" : np.inf,
    "linkage" : 'ward',
    "clustering_threshold" : 0.5,
    "max_iter" : 1000
}

In [18]:
import numpy as np
import time

def ml_worker(alg, samples):

    predictions = []
    total_time = 0

    for sample in samples:
    
        X_data = sample["RA"].values
        Y_data = sample["DEC"].values
        
        data = np.column_stack((X_data, Y_data))
        
        data = np.asarray(data, dtype=np.float64)

        start_time = time.perf_counter()
        labels, probs, c = algorithms[alg](data, params)
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time
        
        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
            .str.replace(r"\..*$", "", regex=True)  # remove anything after the first dot
        )

        fof_id = str(sample["firstHaloInFOFGroupId"].iloc[0])
        labels = np.insert(labels.astype(str), 0, fof_id)

        predictions.append(labels)
    
    return total_time, predictions

In [ ]:
from ds_plus import milaDS
import astro_utils as au
import numpy as np
import time
from astropy.stats import biweight_location

def dsp_worker(samples):

    predictions = []
    total_time = 0

    for sample in samples:
        
        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)
        
        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)

        X_Kpc, Y_Kpc = au.gal_Mpc_coords(X_data, Y_data, Z_data, x_cluster, y_cluster)
        X_Kpc *= 1000
        Y_Kpc *= 1000
        
        V_data = au.los_vel(Z_data, Z_clus)

        start_time = time.perf_counter()
        galaxy_info, grouping, summary = milaDS.DSp_groups(X_data, Y_data, V_data, Z_clus)
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time
        
        labels = np.array([row[8] for row in grouping]) #9th column corresponds to the group

        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
            .str.replace(r"\..*$", "", regex=True)  # remove anything after the first dot
        )

        fof_id = str(sample["firstHaloInFOFGroupId"].iloc[0])
        labels = np.insert(labels.astype(str), 0, fof_id)

        predictions.append(labels)

    return delta_time, predictions



In [ ]:
from calsagos import lagasu
from calsagos import utils
from calsagos import clumberi
from astropy.stats import biweight_location
import numpy as np
import time

from IPython.display import clear_output

def calsagos_worker(samples):
    #- S-PLUS mock cosmology
    H_mock = 67.3
    Omega_L_mock = 0.685
    Omega_m_mock = 0.315

    range_cut_percentage = 0.2
    #-- GENERAL PARAMETERS
    n_galaxies = 4 # -- number of minimum of galaxies that a group or substructure must have

    predictions = []
    total_time = 0

    for sample in samples:

        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)

        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)
        
        cluster_mass = float(sample["log(m_200)"].iloc[0])
        
        range_cuts = int(len(sample)*range_cut_percentage)
        
        start_time = time.perf_counter()
        id_galaxy = indexes = np.arange(len(sample) + 1)
        
        #-- defining the cluster radius
        r200_kpc = utils.calc_radius_finn(cluster_mass, Z_clus, H_mock, Omega_L_mock, Omega_m_mock, "kiloparsec")

        #-- converting the radius in kpc to a radius in angular units
        #-- NOTE: if the user has an estimate of the r200 of the cluster it is not necessary to calculate this quantity
        r200_degree = utils.convert_kpc_to_angular_distance(r200_kpc, Z_clus, H_mock, Omega_m_mock, "degrees") 

        #-- select cluster members
        cluster_members = clumberi.clumberi(id_galaxy, X_data, Y_data, Z_data, Z_clus, x_cluster, y_cluster, range_cuts)

        # -- defining output parameters from clumberi
        id_member = cluster_members[0]
        ra_member = cluster_members[1]
        dec_member = cluster_members[2]
        redshift_member = cluster_members[3]


        #-- estimating the galaxy separation of galaxies in the cluster sample to be used as input in lagasu
        knn_distance = utils.calc_knn_galaxy_distance(ra_member, dec_member, n_galaxies)

        #-- determining the distance to the k-nearest neighbor of each galaxy in the cluster
        knn_galaxy_distance = knn_distance[0]

        try:
            typical_separation = utils.best_eps_dbscan(id_member, knn_galaxy_distance)
        except:
            clear_output(wait=True)
            print("Error ")
            print(id_member)
            print(len(id_member))
            print(knn_distance)
            print(len(knn_distance))

        #-- Assign galaxies to each substructures
        label_candidates = lagasu.lagasu(id_galaxy, X_data, Y_data, Z_data, 
                            range_cuts, typical_separation, n_galaxies, 'euclidean', 'dbscan', 
                            x_cluster, y_cluster, Z_clus, 
                            r200_degree, 'zspec')
        
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time

        #-- defining output parameters from lagasu
        id_candidates = label_candidates[0]
        ra_candidates = label_candidates[1]
        dec_candidates = label_candidates[2]
        redshift_candidates = label_candidates[3]
        label_zcut = label_candidates[4]
        label_final = label_candidates[5]    
    
        
        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
            .str.replace(r"\..*$", "", regex=True)  # remove anything after the first dot
        )

        fof_id = str(sample["firstHaloInFOFGroupId"].iloc[0])
        labels = np.insert(labels.astype(str), 0, fof_id)

        predictions.append(label_final)

    return total_time, predictions

In [11]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_ml_worker(worker, alg, thread_count, iterations, samples):
    results = []
    
    with ProcessPoolExecutor(max_workers=thread_count) as executor:
            futures = [executor.submit(worker, alg, samples) for _ in range(iterations)]
            for future in as_completed(futures):
                results.append(future.result())
    
    return results

In [12]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_worker(worker, thread_count, iterations, samples):
    results = []

    with ProcessPoolExecutor(max_workers=thread_count) as executor:
        futures = [executor.submit(worker, samples) for _ in range(iterations)]

        for future in as_completed(futures):
            results.append(future.result())
    
    return results

In [13]:
import openpyxl
from openpyxl import Workbook
from itertools import zip_longest

def save_xlsx(results, title):
    wb = Workbook()

    sheet1 = wb.active
    sheet1.title = "Execution Times"
    sheet1.append(["Execution Time (s)"])

    for duration, _ in results:
        sheet1.append([round(duration, 3)])

    for idx, (duration, data) in enumerate(results):
        sheet = wb.create_sheet(title=f"Result_{idx + 1}")

        for row in zip_longest(*data, fillvalue=""):
            sheet.append(row)

    # Save Excel file
    wb.save(f"{title}.xlsx")

In [17]:
threads = 4
iterations = 10

for cs in cluster_samples.keys():
    cluster_sample = cluster_samples[cs]

    folder = f"results_{cs}_samples"
    os.makedirs(folder, exist_ok=True)

    for alg in algorithms.keys():
        ml_results = run_ml_worker(ml_worker, alg, threads, iterations, cluster_sample)
        save_xlsx(ml_results, os.path.join(folder,f"ML_{alg}"))

    #dsp_results = run_worker(dsp_worker, threads, iterations, )
    #save_xlsx(dsp_results, os.path.join(folder,f"DSP"))

    #calsagos_results = run_worker(calsagos_worker, threads, iterations)
    #save_xlsx(calsagos_results, os.path.join(folder,f"CALSAGOS"))

## Metrics Calculation

In [4]:
def retag_noise(cluster_list):
    
    new_cluster_list = []
    for df in cluster_list:
        counts = df.groupby("haloId")["haloId"].transform("count")
        df = df.copy()
        df.loc[counts <= 3, "haloId"] = -1
        new_cluster_list.append(df.reset_index(drop=True))
    return new_cluster_list

In [5]:
import my_data_manager as mdm
import pandas as pd

cat = mdm.load_cat("galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)

clean_clusters = [df[df.groupby("haloId")["haloId"].transform("count") > 3].reset_index(drop=True) for df in clusters]

N = 21

splus_clusters = [df[pd.to_numeric(df['mag_r'], errors = 'coerce') >= N].copy() for df in clusters]

clusters = retag_noise(clusters)
splus_clusters = retag_noise(splus_clusters)

cluster_samples = {
    "raw": clusters,
    "clean": clean_clusters,
    "splus": splus_clusters
}

/mnt/c/Users/gasep/OneDrive/Documentos/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/mnt/c/Users/gasep/OneDrive/Documentos/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [1]:
import glob
import os
import pandas as pd

def load_results(sample_type: str):

    folder = f"results_{sample_type}_samples"
    results_df = {}
    
    files = glob.glob(os.path.join(folder, "*.xlsx"))


    for f in files:
        
        xls = pd.ExcelFile(f)
        sheet_names = xls.sheet_names[1:]  # skip the first sheet
        
        key = os.path.splitext(os.path.basename(f))[0]

        df_time = pd.read_excel(xls, sheet_name=sheet_names[0])

        executions = []
        for name in sheet_names:
            df = pd.read_excel(xls, sheet_name=name)
            df.columns = df.columns.map(str)
            executions.append(df)
        
        results_df[key] = {"time": df_time, "executions": executions}

    return results_df

In [6]:
def transpose_list_of_dfs(dfs):
    n_exec = len(dfs)
    n_features = dfs[0].shape[1]

    # Force column labels to string IDs
    for i in range(len(dfs)):
        dfs[i].columns = dfs[i].columns.map(lambda x: str(int(x)) if pd.notna(x) else str(x))

    feature_dfs = {}

    for col_idx in range(n_features):
        feature_id = dfs[0].columns[col_idx]

        col_data = [df.iloc[:, col_idx].reset_index(drop=True) for df in dfs]

        df_new = pd.concat(col_data, axis=1)
        df_new.columns = [f"exec_{i+1}" for i in range(n_exec)]

        feature_dfs[feature_id] = df_new

    return feature_dfs


In [4]:
def do_keys_match(keys1, keys2):

    missing_in_dict = set(keys1) - set(keys2)
    extra_in_dict   = set(keys2) - set(keys1)

    return (len(missing_in_dict) == 0)


In [ ]:
from sklearn.metrics import adjusted_rand_score, v_measure_score, normalized_mutual_info_score
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import cluster_utils as cu
import pickle
import numpy as np


def comput_result_metrics(report, samples, sample_type: str):
    metrics = {}
    for method in report.keys():
        measurements = {}

        alg_report = report[method]
        times = alg_report["time"]
        executions = alg_report["executions"]
        clustering_results = transpose_list_of_dfs(executions)

        keys1 = [str(df["firstHaloInFOFGroupId"].iloc[0]).split(".")[0] for df in samples]
        keys2 = clustering_results.keys()

        if not do_keys_match(keys1, keys2):
            continue

        ari = []
        vmeas = []
        nmi = []
        sil = []
        chs = []
        dbs = []

        dist_matrices = []
        n_completeness_matrices = []
        n_purity_matrices = []
        n_f1_matrices = []

        n_completeness_t_gradients = []
        n_purity_t_gradients = []
        n_f1_t_gradients = []

        n_completeness_p_gradients = []
        n_purity_p_gradients = []
        n_f1_p_gradients = []

        m_completeness_matrices = []
        m_purity_matrices = []
        m_f1_matrices = []

        m_completeness_t_gradients = []
        m_purity_t_gradients = []
        m_f1_t_gradients = []

        m_completeness_p_gradients = []
        m_purity_p_gradients = []
        m_f1_p_gradients = []

        print(f"Computin metrics for method: {method}")

        for galaxy_cluster in samples:
            s = str(galaxy_cluster["firstHaloInFOFGroupId"].iloc[0]).split(".")[0]
            results = clustering_results[s]

            ari_clus = []
            vmeas_clus = []
            nmi_clus = []
            sil_clus = []
            chs_clus = []
            dbs_clus = []

            dist_matrix_clus = []
            n_completeness_matrix_clus = []
            n_purity_matrix_clus = []
            n_f1_matrix_clus = []

            n_completeness_t_gradient_clus = []
            n_purity_t_gradient_clus = []
            n_f1_t_gradient_clus = []

            n_completeness_p_gradient_clus = []
            n_purity_p_gradient_clus = []
            n_f1_p_gradient_clus = []

            m_completeness_matrix_clus = []
            m_purity_matrix_clus = []
            m_f1_matrix_clus = []

            m_completeness_t_gradient_clus = []
            m_purity_t_gradient_clus = []
            m_f1_t_gradient_clus = []

            m_completeness_p_gradient_clus = []
            m_purity_p_gradient_clus = []
            m_f1_p_gradient_clus = []

            for exec in results.columns:

                labels = results[exec].dropna().to_numpy(dtype=np.float64)
                true_labels = galaxy_cluster["haloId"].to_numpy(dtype=np.float64)

                data = galaxy_cluster[["RA", "DEC"]].to_numpy(dtype=np.float64)

                assert len(labels) == len(true_labels), f"Error in cluster {id} for method {method} due to length mismatch.{len(labels)} vs {len(true_labels)}"
                
                ari_clus.append(adjusted_rand_score(true_labels, labels))
                vmeas_clus.append(v_measure_score(true_labels, labels))
                nmi_clus.append(normalized_mutual_info_score(true_labels, labels))
                if len(set(labels)) > 1 and len(set(true_labels)) > 1:
                    sil_clus.append(silhouette_score(data, labels))
                    chs_clus.append(calinski_harabasz_score(data, labels))
                    dbs_clus.append(davies_bouldin_score(data, labels))
                else:
                    sil_clus.append(np.nan)
                    chs_clus.append(np.nan)
                    dbs_clus.append(np.nan)

                distances = cu.calculate_distances(data, true_labels, labels)
                dist_matrix_clus.append(distances)
                n_completeness, n_purity, n_f1_score = cu.calculate_overlap_metrics(data, true_labels, labels)

                n_completeness_matrix_clus.append(n_completeness)
                n_purity_matrix_clus.append(n_purity)
                n_f1_matrix_clus.append(n_f1_score)

                m_completeness, m_purity, m_f1_score = cu.calculate_membership_metrics(true_labels, labels)

                m_completeness_matrix_clus.append(m_completeness)
                m_purity_matrix_clus.append(m_purity)
                m_f1_matrix_clus.append(m_f1_score)

                n_completness_t_gradient = cu.reliability_gradient(n_completeness)
                n_purity_t_gradient = cu.reliability_gradient(n_purity)
                n_f1_t_gradient = cu.reliability_gradient(n_f1_score)

                n_completeness_t_gradient_clus.append(n_completness_t_gradient)
                n_purity_t_gradient_clus.append(n_purity_t_gradient)
                n_f1_t_gradient_clus.append(n_f1_t_gradient)

                n_completness_p_gradient = cu.reliability_gradient(n_completeness, source_axis=0)
                n_purity_p_gradient = cu.reliability_gradient(n_purity, source_axis=0)
                n_f1_p_gradient = cu.reliability_gradient(n_f1_score, source_axis=0)

                n_completeness_p_gradient_clus.append(n_completness_p_gradient)
                n_purity_p_gradient_clus.append(n_purity_p_gradient)
                n_f1_p_gradient_clus.append(n_f1_p_gradient)

                m_completness_t_gradient = cu.reliability_gradient(m_completeness)
                m_purity_t_gradient = cu.reliability_gradient(m_purity)
                m_f1_t_gradient = cu.reliability_gradient(m_f1_score)

                m_completeness_t_gradient_clus.append(m_completness_t_gradient)
                m_purity_t_gradient_clus.append(m_purity_t_gradient)
                m_f1_t_gradient_clus.append(m_f1_t_gradient)

                m_completness_p_gradient = cu.reliability_gradient(m_completeness, source_axis=0)
                m_purity_p_gradient = cu.reliability_gradient(m_purity, source_axis=0)
                m_f1_p_gradient = cu.reliability_gradient(m_f1_score, source_axis=0)

                m_completeness_p_gradient_clus.append(m_completness_p_gradient)
                m_purity_p_gradient_clus.append(m_purity_p_gradient)
                m_f1_p_gradient_clus.append(m_f1_p_gradient)

        
            ari.append(ari_clus)
            vmeas.append(vmeas_clus)
            nmi.append(nmi_clus)
            sil.append(sil_clus)
            chs.append(chs_clus)
            dbs.append(dbs_clus)

            dist_matrices.append(dist_matrix_clus)
            n_completeness_matrices.append(n_completeness_matrix_clus)
            n_purity_matrices.append(n_purity_matrix_clus)
            n_f1_matrices.append(n_f1_matrix_clus)

            n_completeness_t_gradients.append(n_completeness_t_gradient_clus)
            n_purity_t_gradients.append(n_purity_t_gradient_clus)
            n_f1_t_gradients.append(n_f1_t_gradient_clus)

            n_completeness_p_gradients.append(n_completeness_p_gradient_clus)
            n_purity_p_gradients.append(n_purity_p_gradient_clus)
            n_f1_p_gradients.append(n_f1_p_gradient_clus)

            m_completeness_matrices.append(m_completeness_matrix_clus)
            m_purity_matrices.append(m_purity_matrix_clus)
            m_f1_matrices.append(m_f1_matrix_clus)

            m_completeness_t_gradients.append(m_completeness_t_gradient_clus)
            m_purity_t_gradients.append(m_purity_t_gradient_clus)
            m_f1_t_gradients.append(m_f1_t_gradient_clus)

            m_completeness_p_gradients.append(m_completeness_p_gradient_clus)
            m_purity_p_gradients.append(m_purity_p_gradient_clus)
            m_f1_p_gradients.append(m_f1_p_gradient_clus)

        
        avg_time = times.to_numpy().mean()
        std_time = times.to_numpy().std()

        measurements["avg_time"] = avg_time
        measurements["std_time"] = std_time

        measurements["ARI_list"] = ari
        measurements["avg_ARI"] = np.mean(ari)
        measurements["std_ARI"] = np.std(ari)
        measurements["V-meas_list"] = vmeas
        measurements["avg_V-meas"] = np.mean(vmeas)
        measurements["std_V-meas"] = np.std(vmeas)
        measurements["NMI_list"] = nmi
        measurements["avg_NMI"] = np.mean(nmi)
        measurements["std_NMI"] = np.std(nmi)
        measurements["Silhouette_list"] = sil
        measurements["avg_Silhouette"] = np.nanmean(sil)
        measurements["std_Silhouette"] = np.nanstd(sil)
        measurements["CHS_list"] = chs
        measurements["avg_CHS"] = np.nanmean(chs)
        measurements["std_CHS"] = np.nanstd(chs)
        measurements["DBS_list"] = dbs
        measurements["avg_DBS"] = np.nanmean(dbs)
        measurements["std_DBS"] = np.nanstd(dbs)

        measurements["Dist_matrix_list"] = dist_matrices
        measurements["N_Completeness_list"] = n_completeness_matrices
        measurements["N_purity_list"] = n_purity_matrices
        measurements["N_F1_list"] = n_f1_matrices

        measurements["N_Completeness_t_gradient_list"] = n_completeness_t_gradients
        measurements["N_Completeness_t_gradient_avg"] = np.mean(np.vstack(n_completeness_t_gradients), axis=0)
        measurements["N_Completeness_t_gradient_std"] = np.std(np.vstack(n_completeness_t_gradients), axis=0)

        measurements["N_Purity_t_gradient_list"] = n_purity_t_gradients
        measurements["N_Purity_t_gradient_avg"] = np.mean(np.vstack(n_purity_t_gradients), axis=0)
        measurements["N_Purity_t_gradient_std"] = np.std(np.vstack(n_purity_t_gradients), axis=0)
        
        measurements["N_F1_t_gradient_list"] = n_f1_t_gradients
        measurements["N_F1_t_gradient_avg"] = np.mean(np.vstack(n_f1_t_gradients), axis=0)
        measurements["N_F1_t_gradient_std"] = np.std(np.vstack(n_f1_t_gradients), axis=0)

        measurements["N_Completeness_p_gradient_list"] = n_completeness_p_gradients
        measurements["N_Completeness_p_gradient_avg"] = np.mean(np.vstack(n_completeness_p_gradients), axis=0)
        measurements["N_Completeness_p_gradient_std"] = np.std(np.vstack(n_completeness_p_gradients), axis=0)

        measurements["N_Purity_p_gradient_list"] = n_purity_p_gradients
        measurements["N_Purity_p_gradient_avg"] = np.mean(np.vstack(n_purity_p_gradients), axis=0)
        measurements["N_Purity_p_gradient_std"] = np.std(np.vstack(n_purity_p_gradients), axis=0)

        measurements["N_F1_p_gradient_list"] = n_f1_p_gradients
        measurements["N_F1_p_gradient_avg"] = np.mean(np.vstack(n_f1_p_gradients), axis=0)
        measurements["N_F1_p_gradient_std"] = np.std(np.vstack(n_f1_p_gradients), axis=0)

        measurements["M_Completeness_list"] = m_completeness_matrices
        measurements["M_purity_list"] = m_purity_matrices
        measurements["M_F1_list"] = m_f1_matrices

        measurements["M_Completeness_t_gradient_list"] = m_completeness_t_gradients
        measurements["M_Completeness_t_gradient_avg"] = np.mean(np.vstack(m_completeness_t_gradients), axis=0)
        measurements["M_Completeness_t_gradient_std"] = np.std(np.vstack(m_completeness_t_gradients), axis=0)

        measurements["M_Purity_t_gradient_list"] = m_purity_t_gradients
        measurements["M_Purity_t_gradient_avg"] = np.mean(np.vstack(m_purity_t_gradients), axis=0)
        measurements["M_Purity_t_gradient_std"] = np.std(np.vstack(m_purity_t_gradients), axis=0)

        measurements["M_F1_t_gradient_list"] = m_f1_t_gradients
        measurements["M_F1_t_gradient_avg"] = np.mean(np.vstack(m_f1_t_gradients), axis=0)
        measurements["M_F1_t_gradient_std"] = np.std(np.vstack(m_f1_t_gradients), axis=0)

        measurements["M_Completeness_p_gradient_list"] = m_completeness_p_gradients
        measurements["M_Completeness_p_gradient_avg"] = np.mean(np.vstack(m_completeness_p_gradients), axis=0)
        measurements["M_Completeness_p_gradient_std"] = np.std(np.vstack(m_completeness_p_gradients), axis=0)

        measurements["M_Purity_p_gradient_list"] = m_purity_p_gradients
        measurements["M_Purity_p_gradient_avg"] = np.mean(np.vstack(m_purity_p_gradients), axis=0)
        measurements["M_Purity_p_gradient_std"] = np.std(np.vstack(m_purity_p_gradients), axis=0)

        measurements["M_F1_p_gradient_list"] = m_f1_p_gradients
        measurements["M_F1_p_gradient_avg"] = np.mean(np.vstack(m_f1_p_gradients), axis=0)
        measurements["M_F1_p_gradient_std"] = np.std(np.vstack(m_f1_p_gradients), axis=0)

        metrics[method] = measurements
        
    with open(f"metrics_{sample_type}.pkl", "wb") as f:
        pickle.dump(metrics, f)


In [ ]:
import my_data_manager as mdm

cat = mdm.load_cat("galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)

/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/ClusteringComparison/my_data_manager.py:53: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/ClusteringComparison/my_data_manager.py:53: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [ ]:
retaged_clusters = retag_noise(clusters)

In [2]:
import pandas as pd

def compute_times(reported_results):
    rows = []
    for method, alg_report in reported_results.items():
        times = alg_report["time"]
        avg_time = times.to_numpy().mean()
        std_time = times.to_numpy().std()
        rows.append({"method": method, "avg": avg_time, "std": std_time})

    df = pd.DataFrame(rows, columns=["method", "avg", "std"])
    return df



In [3]:
import pickle
import gc

keys = ["raw", "clean", "splus"]

for key in keys:

    times = compute_times(load_results(key))

    with open(f"{key}_times.pkl", "wb") as f:
        pickle.dump(times, f)
        
    del times
    gc.collect()
gc.collect()

0

In [ ]:
import numpy as np
import pandas as pd

def compute_performance_metric(reported_results, samples, metric_func):
    rows = []
    
    for method in reported_results.keys():

        alg_report = reported_results[method]
        executions = alg_report["executions"]
        clustering_results = transpose_list_of_dfs(executions)

        metric = []

        for galaxy_cluster in samples:
            s = str(galaxy_cluster["firstHaloInFOFGroupId"].iloc[0]).split(".")[0]
            results = clustering_results[s]

            metric_clus = []
            for exec in results.columns:

                labels = results[exec].dropna().to_numpy(dtype=np.float64)
                true_labels = galaxy_cluster["haloId"].to_numpy(dtype=np.float64)

                assert len(labels) == len(true_labels), f"Error in cluster {id} for method {method} due to length mismatch.{len(labels)} vs {len(true_labels)}"
                
                metric_clus.append(metric_func(true_labels, labels))
        
            metric.append(metric_clus)

        rows.append({"method": method, "avg": np.mean(metric), "std": np.std(metric)})

    df = pd.DataFrame(rows, columns=["method", "avg", "std"])  
    return df



In [ ]:
import pickle
import gc
from sklearn.metrics import adjusted_rand_score

keys = ["raw", "clean", "splus"]

for key in keys:

    ARIs = compute_performance_metric(load_results(key), cluster_samples[key], adjusted_rand_score )

    with open(f"{key}_ARI.pkl", "wb") as f:
        pickle.dump(ARIs, f)
        
    del ARIs
    gc.collect()
gc.collect()

KeyError: '33000032000016'

In [ ]:
import pickle
import gc
from sklearn.metrics import v_measure_score

keys = ["raw", "clean", "splus"]

for key in keys:

    Vms = compute_performance_metric(load_results(key), cluster_samples[key], v_measure_score )

    with open(f"{key}_V_measure.pkl", "wb") as f:
        pickle.dump(Vms, f)
        
    del Vms
    gc.collect()
gc.collect()

In [ ]:
import pickle
import gc
from sklearn.metrics import normalized_mutual_info_score

keys = ["raw", "clean", "splus"]

for key in keys:

    NMIs = compute_performance_metric(load_results(key), cluster_samples[key], normalized_mutual_info_score)

    with open(f"{key}_NMI.pkl", "wb") as f:
        pickle.dump(NMIs, f)
        
    del NMIs
    gc.collect()
gc.collect()

## Results Visualization